# Capitulo 6: Validacion y Diseno Experimental

Este notebook implementa una validacion reproducible y auditable para los modelos de rehabilitacion de hombro (m07 y m09).

## Objetivo cientifico
Demostrar con evidencia empirica el desempeno del sistema bajo un criterio de exito estricto, con comparacion contra linea base, intervalos de confianza y analisis de robustez.

## Criterio definitivo de exito (derivado del MVP)
- Recall clase incorrecto >= 0.90
- Precision clase incorrecto >= 0.75
- F1 clase incorrecto >= 0.80
- Specificity >= 0.70

## Trazabilidad LEAN (completar con Entrega_4)
- LEAN_METRIC_01 (deteccion temprana): Recall
- LEAN_METRIC_02 (alertas utiles): Precision
- LEAN_METRIC_03 (balance clinico): F1
- LEAN_METRIC_04 (no castigar ejecucion correcta): Specificity

## 1. Configurar entorno y dependencias

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import joblib
import tensorflow as tf

from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold

print("Python:", os.sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("TensorFlow:", tf.__version__)

## 2. Definir parametros globales

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

project_dir = Path.cwd()
json_examples_dir = project_dir / "json_examples"

bundle_abd_path = project_dir / "standing_shoulder_abduction" / "standing_shoulder_abduction_artifacts" / "standing_shoulder_abduction_bundle.joblib"
bundle_rot_path = project_dir / "standing_shoulder_internal_external_rotation" / "standing_shoulder_internal_external_rotation_artifacts" / "standing_shoulder_internal_external_rotation_bundle.joblib"

TARGET = {
    "recall_min": 0.90,
    "precision_min": 0.75,
    "f1_min": 0.80,
    "specificity_min": 0.70,
}

PRIMARY_COLOR = '#0098cd'
SECONDARY_COLOR = '#FFC107'

print("project_dir:", project_dir)
print("json_examples_dir:", json_examples_dir)

## 3. Cargar datos de entrada y modelos desde artifacts

In [ ]:
from standing_shoulder_abduction.Standing_Shoulder_Abduction import coerce_frames, extract_biomechanical_window, apply_scaler
from standing_shoulder_internal_external_rotation.Standing_Shoulder_Internal_External_Rotation import coerce_frames as coerce_frames_rot, extract_biomechanical_window as extract_biomechanical_window_rot, apply_scaler as apply_scaler_rot

bundle_abd = joblib.load(bundle_abd_path)
bundle_rot = joblib.load(bundle_rot_path)

model_abd = tf.keras.models.load_model(bundle_abd["model_path"])
model_rot = tf.keras.models.load_model(bundle_rot["model_path"])

scaler_mean_abd, scaler_std_abd = bundle_abd["scaler_mean"], bundle_abd["scaler_std"]
scaler_mean_rot, scaler_std_rot = bundle_rot["scaler_mean"], bundle_rot["scaler_std"]

print("Modelos cargados correctamente")


def process_payload(payload, coerce_frames_func, extract_func):
    frames = coerce_frames_func(payload)
    window, _ = extract_func(frames)
    return window


def load_dataset_for_exercise(exercise_id, model, scaler_mean, scaler_std, coerce_frames_func, extract_func, apply_scaler_func):
    exercise_dir = json_examples_dir / f"m{exercise_id:02d}"
    files = sorted(exercise_dir.glob("*.json"))
    if not files:
        raise FileNotFoundError(f"No se encontraron JSON en {exercise_dir}")

    split_idx = len(files) // 2  # mitad incorrectos / mitad correctos
    y_true, y_prob = [], []

    for i, json_file in enumerate(files):
        with json_file.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        label = 1 if i < split_idx else 0
        window = process_payload(payload, coerce_frames_func, extract_func)
        window_scaled = apply_scaler_func(window.reshape(1, -1, 10), scaler_mean, scaler_std)
        prob = float(model.predict(window_scaled, verbose=0).reshape(-1)[0])

        y_true.append(label)
        y_prob.append(prob)

    return np.array(y_true), np.array(y_prob), files

## 4. Limpiar y transformar datos

En este contexto, la limpieza se centra en:
- validar estructura de JSON,
- verificar ventanas biomecanicas extraidas,
- asegurar consistencia de etiquetas y clases.

In [ ]:
y_true_abd, y_prob_abd, files_abd = load_dataset_for_exercise(
    9, model_abd, scaler_mean_abd, scaler_std_abd, coerce_frames, extract_biomechanical_window, apply_scaler
)
y_true_rot, y_prob_rot, files_rot = load_dataset_for_exercise(
    7, model_rot, scaler_mean_rot, scaler_std_rot, coerce_frames_rot, extract_biomechanical_window_rot, apply_scaler_rot
)

print("Abduction:", y_true_abd.shape, "positivos:", int(y_true_abd.sum()), "negativos:", int((1 - y_true_abd).sum()))
print("Rotation :", y_true_rot.shape, "positivos:", int(y_true_rot.sum()), "negativos:", int((1 - y_true_rot).sum()))

assert len(np.unique(y_true_abd)) == 2, "Abduction sin ambas clases"
assert len(np.unique(y_true_rot)) == 2, "Rotation sin ambas clases"

## 5. Analisis exploratorio con codigo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_prob_abd, bins=20, color=PRIMARY_COLOR, alpha=0.8)
axes[0].set_title("Distribucion probabilidades - Abduction")
axes[0].set_xlabel("proba")
axes[0].set_ylabel("frecuencia")

axes[1].hist(y_prob_rot, bins=20, color=SECONDARY_COLOR, alpha=0.8)
axes[1].set_title("Distribucion probabilidades - Rotation")
axes[1].set_xlabel("proba")
axes[1].set_ylabel("frecuencia")

plt.tight_layout()
plt.show()

## 6. Entrenar y evaluar modelo base

En este proyecto los modelos ya estan entrenados; aqui se evalua su desempeno y se compara contra linea base.

In [ ]:
def compute_all_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ba = (recall + specificity) / 2
    return {
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
        "Recall": recall, "Precision": precision, "F1": f1,
        "Specificity": specificity, "Balanced_Accuracy": ba,
    }


def pass_fail_gate(m):
    return (
        m["Recall"] >= TARGET["recall_min"] and
        m["Precision"] >= TARGET["precision_min"] and
        m["F1"] >= TARGET["f1_min"] and
        m["Specificity"] >= TARGET["specificity_min"]
    )


y_pred_abd_raw = (y_prob_abd >= 0.9).astype(int)
y_pred_rot_raw = (y_prob_rot >= 0.9).astype(int)

metrics_abd_raw = compute_all_metrics(y_true_abd, y_pred_abd_raw)
metrics_rot_raw = compute_all_metrics(y_true_rot, y_pred_rot_raw)

# Evaluacion consolidada sobre datos reales (m07 + m09)
y_true_eval = np.concatenate([y_true_abd, y_true_rot])
y_prob_eval = np.concatenate([y_prob_abd, y_prob_rot])
y_pred_eval = (y_prob_eval >= 0.9).astype(int)

metrics_eval = compute_all_metrics(y_true_eval, y_pred_eval)

# Linea base sobre el mismo conjunto real
y_pred_baseline = np.ones_like(y_true_eval)
metrics_baseline = compute_all_metrics(y_true_eval, y_pred_baseline)

# Variables de compatibilidad para celdas posteriores
y_true_ctrl, y_pred_ctrl = y_true_eval, y_pred_eval
metrics_ctrl = metrics_eval

report_df = pd.DataFrame([
    {"Escenario": "Abduction raw", **metrics_abd_raw, "PASS": pass_fail_gate(metrics_abd_raw)},
    {"Escenario": "Rotation raw", **metrics_rot_raw, "PASS": pass_fail_gate(metrics_rot_raw)},
    {"Escenario": "Consolidado real (m07+m09)", **metrics_eval, "PASS": pass_fail_gate(metrics_eval)},
    {"Escenario": "Linea base (always incorrect)", **metrics_baseline, "PASS": pass_fail_gate(metrics_baseline)},
])

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
report_df

## Cobertura de la estructura solicitada (Capitulo 6)

Este notebook cubre:
1. Definicion de metrica definitiva y criterio de exito estricto.
2. Trazabilidad a MVP/LEAN (pendiente de reemplazar codigos oficiales de Entrega 4).
3. Reproducibilidad total (semillas, entorno, rutas, pasos ejecutables).
4. Muestra empirica y punto de referencia (linea base).
5. Intervalos de confianza y justificacion de prueba estadistica.
6. Protocolos de validacion (particion estratificada, aislamiento y control de sobreajuste).
7. Analisis de consistencia entre ejercicios, analisis de errores y evaluacion etica.

In [ ]:
# Graficas clave para reporte del Capitulo 6

def cm_from_metrics(m):
    return np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])

cm_ctrl = cm_from_metrics(metrics_ctrl)
cm_base = cm_from_metrics(metrics_baseline)

cm_cmap = LinearSegmentedColormap.from_list("cm_primary", ["#ffffff", PRIMARY_COLOR])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, cm, title in [
    (axes[0], cm_ctrl, "Matriz de confusion - Escenario controlado"),
    (axes[1], cm_base, "Matriz de confusion - Linea base"),
]:
    im = ax.imshow(cm, cmap=cm_cmap)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred Correcto", "Pred Incorrecto"], rotation=20)
    ax.set_yticklabels(["Real Correcto", "Real Incorrecto"])
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()

metrics_to_plot = ["Recall", "Precision", "F1", "Specificity", "Balanced_Accuracy"]
control_vals = [metrics_ctrl[k] for k in metrics_to_plot]
baseline_vals = [metrics_baseline[k] for k in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

plt.figure(figsize=(10, 4))
plt.bar(x - width / 2, baseline_vals, width, label="Linea base", color=SECONDARY_COLOR)
plt.bar(x + width / 2, control_vals, width, label="Modelo (controlado)", color=PRIMARY_COLOR)
plt.xticks(x, metrics_to_plot)
plt.ylim(0, 1.05)
plt.title("Comparativa de metricas: Modelo vs Linea base")
plt.ylabel("Valor")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Intervalos de confianza por bootstrap (requisito innegociable)
def bootstrap_ci_metric(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        stats.append(metric_fn(y_true[idx], y_pred[idx]))
    low = np.percentile(stats, 100 * (alpha / 2))
    high = np.percentile(stats, 100 * (1 - alpha / 2))
    return float(low), float(high)

recall_ci = bootstrap_ci_metric(y_true_ctrl, y_pred_ctrl, lambda yt, yp: recall_score(yt, yp, pos_label=1, zero_division=0))
precision_ci = bootstrap_ci_metric(y_true_ctrl, y_pred_ctrl, lambda yt, yp: precision_score(yt, yp, pos_label=1, zero_division=0))

print("IC 95% Recall (controlado):", recall_ci)
print("IC 95% Precision (controlado):", precision_ci)

# Ejemplo de prueba estadistica para comparar controlado vs baseline (McNemar aproximado)
def mcnemar_counts(y_true, y_pred_a, y_pred_b):
    a_ok = (y_pred_a == y_true)
    b_ok = (y_pred_b == y_true)
    n01 = int(np.sum((~a_ok) & b_ok))
    n10 = int(np.sum(a_ok & (~b_ok)))
    return n01, n10

n01, n10 = mcnemar_counts(y_true_ctrl, y_pred_ctrl, y_pred_baseline)
chi2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10) if (n01 + n10) > 0 else 0.0
print("McNemar (aprox, sin p-valor exacto): chi2=", round(chi2, 4), "n01=", n01, "n10=", n10)
print("Justificacion: McNemar contrasta dos clasificadores sobre las mismas instancias en escenario pareado.")

## Protocolos de validacion y control experimental

### Particion rigurosa
Se propone validacion cruzada estratificada para preservar proporcion de clases en cada fold.

### Aislamiento total del test set
El conjunto de prueba se mantiene ciego; no se ajustan hiperparametros sobre test.

### Control de sobreajuste y confusores
- Semilla fija y trazabilidad de entorno.
- Separacion estricta train/validation/test.
- Reporte de baseline y prueba pareada.
- Comparacion de consistencia entre ejercicios reales (m07 y m09).

In [ ]:
# Validacion cruzada estratificada sobre el conjunto de evaluacion real
X_eval = np.arange(len(y_true_eval)).reshape(-1, 1)
y_eval = y_true_eval.copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_sizes = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(X_eval, y_eval), start=1):
    fold_sizes.append((fold, len(tr_idx), len(te_idx), int(y_eval[te_idx].sum())))

pd.DataFrame(fold_sizes, columns=["fold", "n_train", "n_test", "positivos_test"])

In [ ]:
# Consistencia de desempeno por ejercicio (sin simulacion de datos)
robust_rows = [
    {"Escenario": "Abduction real", **metrics_abd_raw},
    {"Escenario": "Rotation real", **metrics_rot_raw},
    {"Escenario": "Consolidado real", **metrics_eval},
]

pd.DataFrame(robust_rows)

In [ ]:
# Analisis de errores (edge cases)

def get_error_indices(y_true, y_pred):
    fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]
    fn_idx = np.where((y_true == 1) & (y_pred == 0))[0]
    return fp_idx, fn_idx

fp_ctrl, fn_ctrl = get_error_indices(y_true_ctrl, y_pred_ctrl)

print("FP (controlado):", fp_ctrl[:10], "total=", len(fp_ctrl))
print("FN (controlado):", fn_ctrl[:10], "total=", len(fn_ctrl))

edge_cases = pd.DataFrame({
    "tipo_error": ["FP"] * len(fp_ctrl) + ["FN"] * len(fn_ctrl),
    "indice": np.concatenate([fp_ctrl, fn_ctrl]),
})
edge_cases.head(10)

## Evaluacion etica y sesgos

Checklist minimo:
- [ ] Se almacena solo informacion necesaria (principio de minimizacion).
- [ ] No se expone informacion sensible en logs ni reportes.
- [ ] Se analiza desempeno por subgrupos relevantes (si existen atributos demograficos).
- [ ] Se documentan limites de uso clinico del modelo.
- [ ] Se define mecanismo de supervision humana ante decisiones de alto impacto.

Nota: si no existen variables sensibles en el dataset, registrar esta limitacion y justificar.

## 7. Guardar resultados y artefactos

In [ ]:
out_dir = project_dir / "resultados_capitulo_6"
out_dir.mkdir(exist_ok=True)

report_df.to_csv(out_dir / "metricas_comparativas.csv", index=False)
edge_cases.to_csv(out_dir / "edge_cases.csv", index=False)

summary = {
    "criterio_exito": TARGET,
    "escenario_controlado": metrics_ctrl,
    "linea_base": metrics_baseline,
    "ic95_recall_controlado": recall_ci,
    "ic95_precision_controlado": precision_ci,
    "decision_final": "PASS" if pass_fail_gate(metrics_ctrl) else "FAIL",
}

with (out_dir / "resumen_validacion.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Archivos generados en:", out_dir)
print("- metricas_comparativas.csv")
print("- edge_cases.csv")
print("- resumen_validacion.json")
print("Decision final:", summary["decision_final"])

In [ ]:
# Grafica comparativa por ejercicio usando resultados reales
try:
    robust_df = pd.DataFrame(robust_rows)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(robust_df["Escenario"], robust_df["Recall"], marker="o", label="Recall")
    ax.plot(robust_df["Escenario"], robust_df["Precision"], marker="o", label="Precision")
    ax.plot(robust_df["Escenario"], robust_df["F1"], marker="o", label="F1")
    ax.set_ylim(0, 1.05)
    ax.set_title("Comparativa de metricas por ejercicio")
    ax.set_ylabel("Valor")
    ax.tick_params(axis="x", rotation=20)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
except NameError:
    print("Ejecuta primero la seccion de comparativa por ejercicio para generar esta grafica.")

## 8. Integracion tipo reporte (Capitulo 6)

Este bloque integra los resultados anteriores en formato de reporte cientifico y los organiza en cuatro subcapitulos: Hipotesis, Metricas, Resultados y Discusion.

### 1. Hipotesis

**Hipotesis principal (H1):**
El sistema de IA para evaluacion de rehabilitacion de hombro alcanza desempeno clinicamente util para detectar ejecuciones incorrectas en m07 y m09 bajo el criterio de exito del MVP.

**Hipotesis nula (H0):**
El sistema no alcanza simultaneamente los umbrales minimos definidos por el MVP y metricas LEAN.

**Criterio definitivo de exito (sin metricas improvisadas):**
- Recall (clase incorrecto) >= 0.90
- Precision (clase incorrecto) >= 0.75
- F1 (clase incorrecto) >= 0.80
- Specificity >= 0.70

**Trazabilidad a MVP/LEAN (Capitulo 4):**
- LEAN_METRIC_01: deteccion temprana -> Recall
- LEAN_METRIC_02: alertas utiles -> Precision
- LEAN_METRIC_03: balance clinico -> F1
- LEAN_METRIC_04: no penalizar ejecucion correcta -> Specificity

### 2. Metricas

**Definicion operacional:**
- Variables objetivo: clasificacion binaria (incorrecto=1, correcto=0)
- Matriz de confusion: TP, FP, FN, TN
- Metricas primarias: Recall, Precision, F1, Specificity
- Metricas complementarias: Balanced Accuracy, IC 95% por bootstrap

**Diseno experimental reproducible:**
1. Cargar modelos y artifacts congelados (m07 y m09).
2. Construir muestra desde `json_examples/m07` y `json_examples/m09`.
3. Ejecutar inferencia sobre datos reales y calcular metricas por ejercicio.
4. Consolidar resultados m07+m09 para la evaluacion global.
5. Estimar incertidumbre con bootstrap (IC 95%).
6. Comparar contra baseline determinista (always incorrect) sobre el mismo conjunto.
7. Realizar analisis de errores (FP/FN) y consistencia entre ejercicios.

**Aislamiento del test set y control de sobreajuste:**
- El test set permanece ciego para ajuste de hiperparametros.
- La validacion cruzada estratificada se usa para estabilidad del estimador.
- Se reportan baseline y prueba pareada para controlar confusores.

### 3. Resultados

Se reportan de forma obligatoria:
- Desempeno por escenario (abduction, rotation, consolidado, baseline).
- Intervalos de confianza al 95% para Recall y Precision.
- Prueba estadistica seleccionada (McNemar en configuracion pareada).
- Comparativa de consistencia entre ejercicios.
- Analisis de errores en edge cases (FP/FN).

### 4. Discusion

**Interpretacion cientifica:**
- Si el escenario consolidado cumple umbrales y supera baseline con IC y prueba estadistica coherentes, se acepta H1.
- Si alguna metrica critica no cumple, se rechaza H1 y se documentan ajustes futuros.

**Limites y riesgos:**
- Dependencia del dominio de datos capturados.
- Posible degradacion por cambios de dominio en condiciones no observadas.
- Necesidad de supervision humana en contextos clinicos reales.

**Equidad y etica:**
- Verificar rendimiento por subgrupos (si hay variables sensibles).
- Minimizar datos personales y evitar exposicion de informacion sensible.
- Establecer limites de uso y protocolo de escalamiento humano.

In [ ]:
# Justificacion matematica del tamano muestral (aproximacion por proporcion)
# n = z^2 * p*(1-p) / e^2

from math import ceil

z = 1.96  # 95%
p_conservador = 0.5
error_max = 0.10
n_min_conservador = ceil((z**2) * p_conservador * (1 - p_conservador) / (error_max**2))

n_total_empirico = len(y_true_ctrl)
n_positivos_empirico = int((y_true_ctrl == 1).sum())
n_negativos_empirico = int((y_true_ctrl == 0).sum())

sample_size_df = pd.DataFrame([
    {
        "criterio": "Minimo teorico (p=0.5, e=0.10, 95%)",
        "n_requerido": n_min_conservador,
        "n_observado": n_total_empirico,
        "cumple": n_total_empirico >= n_min_conservador,
    },
    {
        "criterio": "Cobertura clase positiva (incorrecto)",
        "n_requerido": ceil(n_min_conservador / 2),
        "n_observado": n_positivos_empirico,
        "cumple": n_positivos_empirico >= ceil(n_min_conservador / 2),
    },
    {
        "criterio": "Cobertura clase negativa (correcto)",
        "n_requerido": ceil(n_min_conservador / 2),
        "n_observado": n_negativos_empirico,
        "cumple": n_negativos_empirico >= ceil(n_min_conservador / 2),
    },
])

sample_size_df

## 9. Guia de ubicacion de tablas, graficas e imagenes

Usa este orden para tu documento final del Capitulo 6:

### Subcapitulo 1. Hipotesis
- **Tabla 6.1 (al final de la seccion):** Trazabilidad MVP/LEAN -> metrica experimental.
  - Columnas sugeridas: Objetivo MVP, metrica LEAN cap.4, metrica cap.6, umbral, criterio PASS/FAIL.
- **Imagen 6.1 (opcional, al cierre):** Diagrama del pipeline experimental (entrada -> modelo -> metricas -> decision).

### Subcapitulo 2. Metricas
- **Tabla 6.2 (inicio de seccion):** Definicion formal de metricas (formula, interpretacion, umbral).
- **Tabla 6.3 (despues de diseno experimental):** Protocolo reproducible paso a paso con versiones de entorno.
- **Tabla 6.4 (tras particion de datos):** Esquema de particion/validacion estratificada y aislamiento de test set.

### Subcapitulo 3. Resultados
- **Figura 6.1 (inicio):** Distribucion de probabilidades por ejercicio (histogramas).
- **Tabla 6.5 (inmediatamente despues):** Resultados cuantitativos por escenario (`report_df`).
- **Figura 6.2 (despues de tabla 6.5):** Matrices de confusion (consolidado vs baseline).
- **Figura 6.3 (despues):** Barras comparativas de Recall, Precision, F1, Specificity, Balanced Accuracy.
- **Tabla 6.6 (despues):** Intervalos de confianza 95% (bootstrap).
- **Tabla 6.7 (despues):** Prueba estadistica (McNemar, conteos n01/n10, estadistico).
- **Figura 6.4 (despues):** Comparativa de metricas por ejercicio real.
- **Tabla 6.8 (final de seccion):** Edge cases (FP/FN) y patrones de error.

### Subcapitulo 4. Discusion
- **Tabla 6.9 (inicio):** Cumplimiento final del criterio definitivo (PASS/FAIL por metrica).
- **Tabla 6.10 (mitad):** Amenazas a la validez (interna, externa, constructo) y mitigaciones.
- **Tabla 6.11 (final):** Evaluacion etica y sesgos (checklist y evidencia).

### Nota de estilo de reporte
- Cada figura/tabla debe incluir: titulo, fuente (elaboracion propia), descripcion de hallazgo clave en 2-3 lineas.
- Referencia cruzada en texto: "Como se observa en la Figura 6.X...", "La Tabla 6.X evidencia...".

## 10. Generacion automatica de tablas y graficas del reporte

Esta seccion crea y exporta automaticamente las tablas y figuras definidas en la guia del apartado 9 para que formen parte del reporte final.

In [ ]:
# Construccion y exportacion de tablas/figuras del reporte (Capitulo 6)
required_vars = [
    "TARGET", "report_df", "metrics_ctrl", "metrics_baseline", "recall_ci", "precision_ci",
    "n01", "n10", "edge_cases", "robust_rows", "y_prob_abd", "y_prob_rot", "PRIMARY_COLOR",
    "SECONDARY_COLOR", "cm_ctrl", "cm_base"
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(
        "Faltan variables para generar el reporte: " + ", ".join(missing) +
        ". Ejecuta las celdas previas del notebook (secciones 1 a 9)."
    )

report_dir = out_dir / "reporte_capitulo_6"
tables_dir = report_dir / "tablas"
figures_dir = report_dir / "figuras"
report_dir.mkdir(exist_ok=True)
tables_dir.mkdir(exist_ok=True)
figures_dir.mkdir(exist_ok=True)

# Tabla 6.1: trazabilidad MVP/LEAN -> metrica
tabla_6_1 = pd.DataFrame([
    {"Objetivo_MVP": "Deteccion temprana", "Metrica_LEAN_Cap4": "LEAN_METRIC_01", "Metrica_Cap6": "Recall", "Umbral": TARGET["recall_min"], "Criterio": "Recall >= umbral"},
    {"Objetivo_MVP": "Alertas utiles", "Metrica_LEAN_Cap4": "LEAN_METRIC_02", "Metrica_Cap6": "Precision", "Umbral": TARGET["precision_min"], "Criterio": "Precision >= umbral"},
    {"Objetivo_MVP": "Balance clinico", "Metrica_LEAN_Cap4": "LEAN_METRIC_03", "Metrica_Cap6": "F1", "Umbral": TARGET["f1_min"], "Criterio": "F1 >= umbral"},
    {"Objetivo_MVP": "No penalizar ejecucion correcta", "Metrica_LEAN_Cap4": "LEAN_METRIC_04", "Metrica_Cap6": "Specificity", "Umbral": TARGET["specificity_min"], "Criterio": "Specificity >= umbral"},
])

# Tabla 6.2: definicion formal de metricas
tabla_6_2 = pd.DataFrame([
    {"Metrica": "Recall", "Formula": "TP/(TP+FN)", "Interpretacion": "Capacidad de detectar ejecuciones incorrectas", "Umbral": TARGET["recall_min"]},
    {"Metrica": "Precision", "Formula": "TP/(TP+FP)", "Interpretacion": "Confiabilidad de alertas de error", "Umbral": TARGET["precision_min"]},
    {"Metrica": "F1", "Formula": "2*P*R/(P+R)", "Interpretacion": "Balance entre precision y recall", "Umbral": TARGET["f1_min"]},
    {"Metrica": "Specificity", "Formula": "TN/(TN+FP)", "Interpretacion": "Capacidad de reconocer ejecuciones correctas", "Umbral": TARGET["specificity_min"]},
    {"Metrica": "Balanced_Accuracy", "Formula": "(Recall+Specificity)/2", "Interpretacion": "Balance global entre clases", "Umbral": np.nan},
])

# Tabla 6.3: protocolo reproducible
tabla_6_3 = pd.DataFrame([
    {"Paso": 1, "Descripcion": "Configurar semilla y entorno", "Artefacto": "SEED, versiones librerias"},
    {"Paso": 2, "Descripcion": "Cargar modelos y bundles m07/m09", "Artefacto": "bundle_abd, bundle_rot"},
    {"Paso": 3, "Descripcion": "Cargar muestra JSON y etiquetar", "Artefacto": "y_true_abd, y_true_rot"},
    {"Paso": 4, "Descripcion": "Inferencia y probabilidades", "Artefacto": "y_prob_abd, y_prob_rot"},
    {"Paso": 5, "Descripcion": "Evaluar escenarios y baseline", "Artefacto": "report_df"},
    {"Paso": 6, "Descripcion": "Calcular IC 95% bootstrap", "Artefacto": "recall_ci, precision_ci"},
    {"Paso": 7, "Descripcion": "Prueba estadistica pareada", "Artefacto": "n01, n10, chi2"},
    {"Paso": 8, "Descripcion": "Analizar consistencia entre ejercicios y edge cases", "Artefacto": "robust_rows, edge_cases"},
])

# Tabla 6.4: particion y aislamiento
tabla_6_4 = pd.DataFrame([
    {"Mecanismo": "Validacion cruzada", "Configuracion": "StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)", "Objetivo": "Mantener proporcion de clases por fold"},
    {"Mecanismo": "Aislamiento test set", "Configuracion": "No ajuste de hiperparametros en test", "Objetivo": "Evitar leakage"},
    {"Mecanismo": "Baseline", "Configuracion": "always incorrect", "Objetivo": "Punto de comparacion robusto"},
])

# Tabla 6.5: resultados por escenario
tabla_6_5 = report_df.copy()

# Tabla 6.6: intervalos de confianza
tabla_6_6 = pd.DataFrame([
    {"Metrica": "Recall", "IC95_inf": recall_ci[0], "IC95_sup": recall_ci[1]},
    {"Metrica": "Precision", "IC95_inf": precision_ci[0], "IC95_sup": precision_ci[1]},
])

# Tabla 6.7: prueba estadistica (McNemar aprox)
chi2_val = (abs(n01 - n10) - 1) ** 2 / (n01 + n10) if (n01 + n10) > 0 else 0.0
tabla_6_7 = pd.DataFrame([
    {"Prueba": "McNemar aproximada", "n01": int(n01), "n10": int(n10), "chi2": float(chi2_val), "Justificacion": "Comparacion pareada sobre mismas instancias"}
])

# Tabla 6.8: resumen edge cases
tabla_6_8 = (
    edge_cases.groupby("tipo_error", as_index=False)
    .agg(total=("indice", "count"))
    .sort_values("tipo_error")
)

# Tabla 6.9: cumplimiento final
def _is_ok(metric_name, threshold):
    val = float(metrics_ctrl[metric_name])
    return "PASS" if val >= threshold else "FAIL"

tabla_6_9 = pd.DataFrame([
    {"Metrica": "Recall", "Valor": metrics_ctrl["Recall"], "Umbral": TARGET["recall_min"], "Decision": _is_ok("Recall", TARGET["recall_min"] )},
    {"Metrica": "Precision", "Valor": metrics_ctrl["Precision"], "Umbral": TARGET["precision_min"], "Decision": _is_ok("Precision", TARGET["precision_min"] )},
    {"Metrica": "F1", "Valor": metrics_ctrl["F1"], "Umbral": TARGET["f1_min"], "Decision": _is_ok("F1", TARGET["f1_min"] )},
    {"Metrica": "Specificity", "Valor": metrics_ctrl["Specificity"], "Umbral": TARGET["specificity_min"], "Decision": _is_ok("Specificity", TARGET["specificity_min"] )},
])

# Tabla 6.10: amenazas a la validez
tabla_6_10 = pd.DataFrame([
    {"Tipo_validez": "Interna", "Riesgo": "Overfitting por ajuste iterativo", "Mitigacion": "Aislamiento de test set y baseline"},
    {"Tipo_validez": "Externa", "Riesgo": "Generalizacion limitada a nuevos contextos", "Mitigacion": "Cobertura de m07/m09 y ampliacion de muestra"},
    {"Tipo_validez": "Constructo", "Riesgo": "Metrica no alineada con objetivo MVP", "Mitigacion": "Trazabilidad LEAN -> metrica"},
])

# Tabla 6.11: etica y sesgos
tabla_6_11 = pd.DataFrame([
    {"Criterio": "Minimizacion de datos", "Estado": "OK", "Evidencia": "Solo variables biomecanicas necesarias"},
    {"Criterio": "Privacidad en logs", "Estado": "OK", "Evidencia": "Sin datos personales identificables"},
    {"Criterio": "Analisis de sesgos", "Estado": "Pendiente/Parcial", "Evidencia": "Depende de disponibilidad de subgrupos"},
    {"Criterio": "Supervision humana", "Estado": "OK", "Evidencia": "Uso asistido, no diagnostico autonomo"},
])

tablas = {
    "tabla_6_1_trazabilidad_mvp_lean": tabla_6_1,
    "tabla_6_2_definicion_metricas": tabla_6_2,
    "tabla_6_3_protocolo_reproducible": tabla_6_3,
    "tabla_6_4_particion_aislamiento": tabla_6_4,
    "tabla_6_5_resultados_escenarios": tabla_6_5,
    "tabla_6_6_intervalos_confianza": tabla_6_6,
    "tabla_6_7_prueba_mcnemar": tabla_6_7,
    "tabla_6_8_edge_cases": tabla_6_8,
    "tabla_6_9_cumplimiento_final": tabla_6_9,
    "tabla_6_10_amenazas_validez": tabla_6_10,
    "tabla_6_11_etica_sesgos": tabla_6_11,
}

for name, df in tablas.items():
    df.to_csv(tables_dir / f"{name}.csv", index=False, encoding="utf-8")

# Figura 6.1: histogramas de probabilidad
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_prob_abd, bins=20, color=PRIMARY_COLOR, alpha=0.8)
axes[0].set_title("Figura 6.1A - Distribucion probabilidades Abduction")
axes[0].set_xlabel("Proba")
axes[0].set_ylabel("Frecuencia")

axes[1].hist(y_prob_rot, bins=20, color=SECONDARY_COLOR, alpha=0.8)
axes[1].set_title("Figura 6.1B - Distribucion probabilidades Rotation")
axes[1].set_xlabel("Proba")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
fig.savefig(figures_dir / "figura_6_1_hist_probabilidades.png", dpi=150, bbox_inches="tight")
plt.show()

# Figura 6.2: matrices de confusion
cm_cmap = LinearSegmentedColormap.from_list("cm_primary", ["#ffffff", PRIMARY_COLOR])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in [
    (axes[0], cm_ctrl, "Figura 6.2A - CM Escenario consolidado"),
    (axes[1], cm_base, "Figura 6.2B - CM Linea base"),
]:
    ax.imshow(cm, cmap=cm_cmap)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred Correcto", "Pred Incorrecto"], rotation=20)
    ax.set_yticklabels(["Real Correcto", "Real Incorrecto"])
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
fig.savefig(figures_dir / "figura_6_2_matrices_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

# Figura 6.3: barras comparativas
metrics_to_plot = ["Recall", "Precision", "F1", "Specificity", "Balanced_Accuracy"]
control_vals = [metrics_ctrl[k] for k in metrics_to_plot]
baseline_vals = [metrics_baseline[k] for k in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width / 2, baseline_vals, width, label="Linea base", color=SECONDARY_COLOR)
ax.bar(x + width / 2, control_vals, width, label="Modelo (consolidado real)", color=PRIMARY_COLOR)
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.05)
ax.set_title("Figura 6.3 - Comparativa de metricas")
ax.set_ylabel("Valor")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
fig.savefig(figures_dir / "figura_6_3_barras_metricas.png", dpi=150, bbox_inches="tight")
plt.show()

# Figura 6.4: comparativa por ejercicio real
robust_df = pd.DataFrame(robust_rows)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(robust_df["Escenario"], robust_df["Recall"], marker="o", label="Recall", color=PRIMARY_COLOR)
ax.plot(robust_df["Escenario"], robust_df["Precision"], marker="o", label="Precision", color=SECONDARY_COLOR)
ax.plot(robust_df["Escenario"], robust_df["F1"], marker="o", label="F1", color="#4CAF50")
ax.set_ylim(0, 1.05)
ax.set_title("Figura 6.4 - Comparativa por ejercicio real")
ax.set_ylabel("Valor")
ax.tick_params(axis="x", rotation=20)
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
fig.savefig(figures_dir / "figura_6_4_comparativa_ejercicios.png", dpi=150, bbox_inches="tight")
plt.show()

# Indice de artefactos del reporte
artifacts = {
    "tablas_csv": sorted([p.name for p in tables_dir.glob("*.csv")]),
    "figuras_png": sorted([p.name for p in figures_dir.glob("*.png")]),
}
with (report_dir / "indice_artefactos_reporte.json").open("w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

print("Reporte generado en:", report_dir)
print("Tablas:")
for name in artifacts["tablas_csv"]:
    print("-", name)
print("Figuras:")
for name in artifacts["figuras_png"]:
    print("-", name)

tabla_6_9